# SDXL inpainting starter notebook (room generation — exploratory scaffold)

**Model note, corrected from the original ask:** this uses **SDXL
inpainting** (`diffusers/stable-diffusion-xl-1.0-inpainting-0.1`), not Qwen.
`ADR-0020` (`architecture/adr/ADR-0020-real-room-generation-model-and-adapter-architecture.md`)
ruled on this directly: Qwen-Image-Edit-2509 was evaluated and rejected as
the *primary* backend on hardware (~40GB of weights, no funded GPU) and API
shape (no native `mask_image`/`strength` dial, which a low-denoise seam
blend needs) — it remains a documented comparison arm (phase P6) if GPU
hardware is ever funded. SDXL wins on API shape, not image quality.

**What this notebook is.** An exploratory scaffold for learning the
`diffusers` inpainting API and trying the shape of the real pipeline
(load model -> inpaint a masked region -> paste real product pixels back on
top). It is deliberately simple, per the "table it if it's too complex for
now" instruction.

**What this notebook is not, and what still has to happen first.** This is
*not* the real room-generation service, and running it does not unblock
anything in `agent_instructions/STATUS.md` item 27. Per `ADR-0020`, three
CPU-only prerequisites (phases P1-P3) have to land in
`ai_services/room_generator/` *before* any of this matters for real:
rooms currently can't store image bytes at all (only a reference string),
its worker never calls a generation adapter, and there is no
byte-exactness compositing test yet proving product pixels survive
untouched. This notebook skips straight to "does the model call work at
all", which is a fine thing to explore in parallel, but it is not a
substitute for that CPU-only foundation work.

**Hardware.** SDXL inpainting needs a real GPU (`ADR-0020`: ~12GB VRAM) —
this will not run practically on a CPU-only laptop. Treat the cells below
as something to *run in Colab with a GPU runtime*, not locally, unless your
machine has one.

## Running locally vs. in Google Colab

**Colab (the realistic option for this notebook):** `Runtime > Change
runtime type > GPU`, then:

```python
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = Path('/content/drive/MyDrive/Supplier Images')
```

**Local (default below):** `DATA_DIR` points at the product photography
already on disk. Model loading is gated behind `RUN_HEAVY_CELLS = False`
by default (see below) so opening this notebook doesn't silently try to
download several GB of weights or fail loudly on a machine with no GPU.


In [ ]:
# Uncomment in Colab (locally, install these once from your terminal instead):
# !pip install diffusers transformers accelerate torch pillow

import os
from pathlib import Path

from PIL import Image

# --- Path configuration --------------------------------------------------
DATA_DIR = Path(os.environ.get(
    "CURALINA_SUPPLIER_IMAGES_DIR",
    "/Users/rjsalmon/Downloads/Supplier Images",
))

# ATRIANI and LUXUS are white-background product photography -- the only
# two suppliers with cutout-friendly images today (confirmed by inspecting
# the folders directly: Celadon is one flat artwork image per SKU, Lazzoni
# has no product photos at all, only PDF spec sheets).
PRODUCT_IMAGE_DIRS = {
    "ATRIANI": DATA_DIR / "ATRIANI",
    "LUXUS": DATA_DIR / "LUXUS" / "Product Images",
}
for supplier, path in PRODUCT_IMAGE_DIRS.items():
    print(supplier, "->", path, "exists:", path.exists())

# Set this to True only on a machine/runtime with a real GPU.
RUN_HEAVY_CELLS = False
MODEL_ID = "diffusers/stable-diffusion-xl-1.0-inpainting-0.1"


## 1. Pick one product cutout to work with

Just to have something concrete to look at before touching the model —
list what's actually in one product's folder and open the first image.


In [ ]:
example_supplier = "LUXUS"
example_dir = PRODUCT_IMAGE_DIRS[example_supplier]
if example_dir.exists():
    product_folders = sorted(p for p in example_dir.iterdir() if p.is_dir())
    print(f"{len(product_folders)} products under {example_supplier}")
    if product_folders:
        first_product = product_folders[0]
        images = sorted(first_product.glob("*.png")) + sorted(first_product.glob("*.webp"))
        print(f"{first_product.name}: {[i.name for i in images]}")
        if images:
            example_image = Image.open(images[0])
            example_image


## 2. Load the model (GPU only — gated behind `RUN_HEAVY_CELLS`)

This is the actual `diffusers` call. It's wrapped in `if RUN_HEAVY_CELLS`
so importing/reading this notebook doesn't try to download ~7GB of weights
or crash on a CPU-only machine. Flip the flag above to `True` once you're
on a GPU runtime and actually want to run it.


In [ ]:
pipe = None

if RUN_HEAVY_CELLS:
    import torch
    from diffusers import AutoPipelineForInpainting

    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cpu":
        print("WARNING: no GPU detected -- this will be extremely slow or may not fit in memory.")

    pipe = AutoPipelineForInpainting.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    ).to(device)
    print(f"Loaded {MODEL_ID} on {device}")
else:
    print("RUN_HEAVY_CELLS is False -- skipping model load. Set it to True on a GPU runtime.")


## 3. A minimal inpaint call, with the product pixels pasted back on top

This is the teaching-simplified version of `ADR-0020`'s three-stage design
(backplate -> photoreal shell -> composite): here we skip stage 1 (a real
projected backplate) and just inpaint directly into a plain background
image, so the mechanics are visible without needing the rules-engine
geometry code. The one non-negotiable part we keep is the **last step**:
after the model runs, we paste the *original, untouched* product image
back over its own region. That's the same `hard_composite` guarantee
`ADR-0020`/`VAR-A3-03` require in the real service — the model may
hallucinate anything it wants in the background, but the product pixels
customers are shown are never touched by it.


In [ ]:
def inpaint_with_product_preserved(pipe, backplate: Image.Image, mask: Image.Image,
                                    product_cutout: Image.Image, paste_box: tuple[int, int, int, int],
                                    prompt: str, seed: int = 42):
    """Runs one inpaint call, then hard-composites the product back on top.

    `mask` is white where the model is allowed to paint, black everywhere
    else (standard diffusers inpainting convention). `paste_box` is the
    (left, top, right, bottom) region `product_cutout` belongs in.
    """
    import torch

    generator = torch.Generator(device=pipe.device).manual_seed(seed)
    result = pipe(
        prompt=prompt,
        image=backplate,
        mask_image=mask,
        strength=0.8,
        generator=generator,
    ).images[0]

    # The guarantee: paste the ORIGINAL product pixels back, unchanged,
    # regardless of what the model painted underneath them.
    result.paste(product_cutout, paste_box[:2])
    return result


if RUN_HEAVY_CELLS and pipe is not None:
    backplate = Image.new("RGB", (1024, 1024), color=(200, 195, 185))
    mask = Image.new("L", (1024, 1024), color=255)  # paint everywhere for this toy example
    product = example_image.convert("RGBA") if "example_image" in dir() else None
    if product is not None:
        output = inpaint_with_product_preserved(
            pipe, backplate, mask, product,
            paste_box=(400, 500, 700, 900),
            prompt="a bright, airy living room, organic modern style, soft natural light",
        )
        output


## Next steps (not in this notebook)

- `ADR-0020` phases **P1-P3** (CPU-only, no model): asset byte storage for
  rooms, wiring the worker to actually call a generation adapter, and a
  real deterministic backplate projected from room geometry — none of
  these exist in `ai_services/room_generator/` yet, and they're the
  actual prerequisite for any of this mattering in the live service.
- The byte-exactness composite test (`ADR-0020`, mirroring `VAR-A3-03`):
  read the output image back to a numpy array and assert the pasted
  product region is pixel-identical to the source cutout, for every
  backend, not just eyeballed like the toy example above.
- Confirming the SKU join: do ATRIANI/LUXUS image filenames actually
  resolve to the `supplier_sku` values recommendation returns? Nobody has
  checked this yet (flagged in `ADR-0020`'s item 27 guidance) — if not,
  mode S is artwork-only for now (Celadon), which is fine and honest.
- Any accept/reject claim about output quality is `ai-ml-lead`'s call,
  following the same evidence-notebook process as `R01`-`R03`, not
  something this exploratory notebook can establish on its own.
